In [1]:

# M10-A1: Food Delivery Analytics — Full EDA Project

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. GENERATE SYNTHETIC DATASET
# ==========================================

np.random.seed(7)
n = 200

df = pd.DataFrame({
    "restaurant": np.random.choice(
        ["Pizza Hub", "Burger Point", "Spice House",
         "Food Corner", "Tasty Bites"], n
    ),
    "order_value": np.random.uniform(100, 800, n),
    "distance_km": np.random.uniform(1, 20, n),
    "delivery_time_mins": np.random.normal(30, 8, n),
    "rating": np.random.uniform(1.0, 5.0, n),
    "discount_pct": np.random.uniform(0, 30, n)
})

# Ensure delivery time is positive
df["delivery_time_mins"] = df["delivery_time_mins"].clip(lower=5)

print("Original dataset:")
print(df.head())
print("\nDataset shape:", df.shape)


# ==========================================
# 2. INTRODUCE 5% NULL VALUES
# ==========================================

for col in ["delivery_time_mins", "rating"]:
    missing_rows = np.random.choice(
        df.index,
        size=int(0.05 * len(df)),
        replace=False
    )
    df.loc[missing_rows, col] = np.nan

print("\nMissing values before cleaning:")
print(df[["delivery_time_mins", "rating"]].isnull().sum())


# ==========================================
# 3. FILL NULL VALUES USING MEDIAN
# ==========================================

for col in ["delivery_time_mins", "rating"]:
    df[col] = df[col].fillna(df[col].median())

print("\nMissing values after cleaning:")
print(df.isnull().sum())


# ==========================================
# 4. CALCULATE DELIVERY SPEED
# ==========================================

df["delivery_speed_kmph"] = (
    df["distance_km"] / (df["delivery_time_mins"] / 60)
)

# Divide speed into 3 equal-frequency groups
df["speed_band"] = pd.qcut(
    df["delivery_speed_kmph"],
    q=3,
    labels=["Slow", "Normal", "Fast"]
)

print("\nUpdated dataset:")
print(df.head())

print("\nSpeed band counts:")
print(df["speed_band"].value_counts())


# ==========================================
# 5. CORRELATION HEATMAP
# ==========================================

def create_correlation_heatmap():
    numeric_df = df.select_dtypes(include="number")
    corr = numeric_df.corr(method="pearson")

    plt.figure(figsize=(10, 7))

    sns.heatmap(
        corr,
        annot=True,
        cmap="coolwarm",
        fmt=".2f",
        center=0
    )

    plt.title("Food Delivery Pearson Correlation Heatmap")
    plt.tight_layout()
    plt.savefig("correlation_heatmap2.png", dpi=200)
    plt.close()

    print("Saved correlation_heatmap2.png")


# ==========================================
# 6. SEABORN PAIRPLOT
# ==========================================

def create_pairplot():
    graph = sns.pairplot(
        df,
        vars=[
            "order_value",
            "distance_km",
            "delivery_time_mins",
            "rating"
        ],
        hue="speed_band",
        diag_kind="hist"
    )

    graph.fig.suptitle(
        "Food Delivery Analysis by Speed Band",
        y=1.02
    )

    graph.savefig("pairplot2.png", dpi=200)
    plt.close("all")

    print("Saved pairplot2.png")


# ==========================================
# 7. OPTION 1: SUMMARY STATISTICS
# ==========================================

def summary_statistics():
    print("\n--- SUMMARY STATISTICS ---")

    numeric_df = df.select_dtypes(include="number")

    print(numeric_df.describe().round(2))

    print("\nNumPy means:")
    for col in numeric_df.columns:
        print(col, ":", round(np.mean(df[col]), 2))

    numeric_df.describe().to_csv("summary_statistics2.csv")
    print("\nSaved summary_statistics.csv")


# ==========================================
# 8. OPTION 2: DISTRIBUTION ANALYSIS
# ==========================================

def distribution_analysis():
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    axes[0].hist(df["order_value"], bins=10)
    axes[0].set_title("Order Value Distribution")
    axes[0].set_xlabel("Order Value (Rs)")
    axes[0].set_ylabel("Number of Orders")

    axes[1].hist(df["delivery_time_mins"], bins=10)
    axes[1].set_title("Delivery Time Distribution")
    axes[1].set_xlabel("Delivery Time (minutes)")
    axes[1].set_ylabel("Number of Orders")

    plt.tight_layout()
    plt.savefig("distribution_analysis2.png", dpi=200)
    plt.close(fig)

    print("Saved distribution_analysis2.png")


# ==========================================
# 9. OPTION 3: CORRELATION HEATMAP
# ==========================================

def correlation_heatmap_menu():
    create_correlation_heatmap()


# ==========================================
# 10. OPTION 4: RESTAURANT PERFORMANCE
# ==========================================

def restaurant_performance():
    report = df.groupby("restaurant").agg(
        mean_rating=("rating", "mean"),
        total_orders=("rating", "count"),
        average_order_value=("order_value", "mean"),
        average_delivery_time=("delivery_time_mins", "mean")
    ).round(2)

    print("\n--- RESTAURANT PERFORMANCE REPORT ---")
    print(report)

    fig, ax = plt.subplots(figsize=(9, 5))

    report["mean_rating"].sort_values(ascending=False).plot(
        kind="bar",
        ax=ax
    )

    ax.set_title("Average Restaurant Ratings")
    ax.set_xlabel("Restaurant")
    ax.set_ylabel("Mean Rating")

    plt.tight_layout()
    plt.savefig("restaurant_performance.png", dpi=200)
    plt.close(fig)

    print("Saved restaurant_performance.png")


# ==========================================
# 11. FINAL SUMMARY REPORT
# ==========================================

def final_report():
    print("\n========== FINAL EDA REPORT ==========")

    # Top 3 restaurants by mean rating
    top3 = (
        df.groupby("restaurant")["rating"]
        .mean()
        .sort_values(ascending=False)
        .head(3)
    )

    print("\nTop 3 Restaurants by Mean Rating:")
    print(top3.round(2).to_string())

    # Find numeric pair with highest absolute correlation
    corr = df.select_dtypes(include="number").corr()

    upper = corr.where(
        np.triu(np.ones(corr.shape), k=1).astype(bool)
    )

    pairs = upper.abs().stack()

    if not pairs.empty:
        best_pair = pairs.idxmax()

        print("\nHighest Absolute Pearson Correlation:")
        print(
            best_pair[0], "and", best_pair[1],
            "=", round(
                corr.loc[best_pair[0], best_pair[1]], 3
            )
        )

    # Delivery time statistics
    print(
        "\nMean Delivery Time:",
        round(df["delivery_time_mins"].mean(), 2),
        "minutes"
    )

    print(
        "Standard Deviation:",
        round(df["delivery_time_mins"].std(), 2),
        "minutes"
    )


# ==========================================
# 12. MENU-DRIVEN CONSOLE
# ==========================================

while True:
    print("\n===== FOOD DELIVERY ANALYTICS =====")
    print("1. Summary Statistics")
    print("2. Distribution Analysis")
    print("3. Correlation Heatmap")
    print("4. Restaurant Performance Report")
    print("5. Create Pairplot")
    print("6. Exit")

    choice = input("Enter your choice (1-6): ").strip()

    if choice == "1":
        summary_statistics()

    elif choice == "2":
        distribution_analysis()

    elif choice == "3":
        correlation_heatmap_menu()

    elif choice == "4":
        restaurant_performance()

    elif choice == "5":
        create_pairplot()

    elif choice == "6":
        final_report()
        print("\nThank you for using Food Delivery Analytics!")
        break

    else:
        print("Invalid choice. Please enter a number from 1 to 6.")


Original dataset:
     restaurant  order_value  distance_km  delivery_time_mins    rating  \
0   Tasty Bites   269.122042    10.445448           29.497731  2.640619   
1  Burger Point   107.405935     4.055690           26.561146  3.800554   
2   Food Corner   681.448228    14.946773           16.707794  1.787220   
3   Food Corner   748.629056    16.350346           30.647936  1.332268   
4   Tasty Bites   421.022436    14.279122           23.365020  4.726010   

   discount_pct  
0      0.136464  
1     18.496504  
2      8.397846  
3     10.516892  
4     29.078367  

Dataset shape: (200, 6)

Missing values before cleaning:
delivery_time_mins    10
rating                10
dtype: int64

Missing values after cleaning:
restaurant            0
order_value           0
distance_km           0
delivery_time_mins    0
rating                0
discount_pct          0
dtype: int64

Updated dataset:
     restaurant  order_value  distance_km  delivery_time_mins    rating  \
0   Tasty Bites   26